### What is Langgraph
LangGraph is a framework built on top of LangChain for creating stateful, multi-step AI applications using a graph-based workflow. It allows developers to define nodes (tasks), edges (flow), and state (shared data) to build reliable AI agents and complex workflows.

### Build A Basic Chatbot With Langgraph(GRAPH API)

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages:Annotated[list,add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:llama-3.3-70b-versatile")
llm

In [ ]:
llm.invoke("what is Langgraph")

In [ ]:
## Node Functionality
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder=StateGraph(State)

# Adding Nodes
graph_builder.add_node("llmChatbot",chatbot)
# Adding Edges
graph_builder.add_edge(START,"llmChatbot")
graph_builder.add_edge("llmChatbot",END)

# Compile the graph
graph = graph_builder.compile()
graph

In [ ]:
response=graph.invoke({"messages":"Hi I am Sachin"})
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages":"Today News"}):
    for value in event.values():
        print(value["messages"][-1].content)

### Chatbot with Toll

In [ ]:
from langchain_tavily import TavilySearch

tool=TavilySearch(max_results=2)
tool.invoke("What is langgraph")

In [ ]:
## Custom function
def multiply(a:int,b:int)->int:
    """Multiply a and b

    Args:
        a (int): first int
        b (int): second int

    Returns:
        int: output int
    """
    return a*b

In [ ]:
tools=[tool,multiply]

In [ ]:
llm_with_tool=llm.bind_tools(tools)

In [ ]:
llm_with_tool

In [ ]:
## Stategraph
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

## Node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

## Grpah
builder=StateGraph(State)
builder.add_node("tool_calling_llm",tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

## Add Edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools",END)

## compile the graph
graph=builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))